通过深度学习框架来简洁地实现线性回归生成数据集

In [82]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [83]:
true_w=torch.tensor([2,-3.4])
true_b=4.2
features,labels=d2l.synthetic_data(true_w, true_b,num_examples=1000)

## d2l.synthetic_data 函数

**功能**：生成带噪声的合成线性回归数据集。

**参数**：
- `w`：真实权重向量，例如 `[2, -3.4]`。
- `b`：真实偏差，例如 `4.2`。
- `num_examples`：样本数量，例如 `1000`。

**生成规则**：
- 特征 `X`：从标准正态分布采样，形状为 `(num_examples, len(w))`。
- 标签 `y`：满足 `y = Xw + b + 噪声`，噪声为标准差0.01的高斯噪声。
- 返回 `X` 和列向量形式的 `y`。

**用途**：快速构造用于测试线性回归模型和优化算法的数据。

## 数据加载器封装

**函数作用**  
将特征和标签张量组合成数据集，并提供按批次迭代的能力。

**关键组件**  
- `TensorDataset(*data_arrays)`：将多个张量按样本维一一配对，形成数据集。  
  `dataset[i]` 返回第 i 个样本的 `(特征, 标签)`。
- `DataLoader(dataset, batch_size, shuffle)`：  
  - 按 `batch_size` 切分数据集。  
  - `shuffle=True` 时每个 epoch 随机打乱样本顺序；`False` 时保持原顺序。

**使用示例**  
`data_iter = load_array((features, labels), batch_size, is_train=True)`  
之后可被 `for x, y in data_iter` 迭代。

**快速查看一批数据**  
`next(iter(data_iter))` 取出第一个批次，用于检查形状和值。

In [84]:
def load_array(data_arrays,batch_size,is_train=True):
    dataset=data.TensorDataset(*data_arrays) #解包操作，让 features 和 labels 分别给 dataset
    return data.DataLoader(dataset=dataset,batch_size=batch_size,shuffle=is_train) #随机打乱
batch_size=10
data_iter=load_array((features,labels),batch_size,is_train=True) #把 (features, labels) 封装成数据加载器，并赋给 data_iter

next(iter(data_iter))
#iter(data_iter)：把数据加载器转换为迭代器。
#next(...)：取出迭代器的下一个元素，也就是第一个批次的数据。

[tensor([[-0.9800,  2.7731],
         [-1.5243, -0.3159],
         [-0.0058, -0.5865],
         [-1.9937, -0.6816],
         [-0.1298,  1.8107],
         [-0.2827,  2.1692],
         [-0.9575, -1.1405],
         [-0.0298, -0.0352],
         [ 0.7434, -0.5170],
         [ 0.2946, -0.2826]]),
 tensor([[-7.1875],
         [ 2.2162],
         [ 6.1811],
         [ 2.5333],
         [-2.2122],
         [-3.7370],
         [ 6.1570],
         [ 4.2447],
         [ 7.4474],
         [ 5.7417]])]

使用框架地预定好的层

## 定义线性回归模型（PyTorch 方式）

- `nn.Sequential`：容器，按顺序存放网络层，数据依次通过。
- `nn.Linear(in_features, out_features)`：全连接层，做线性变换。
  - 第一参数：输入特征数，例如 `2`。
  - 第二参数：输出特征数，例如 `1`。
- `net = nn.Sequential(nn.Linear(2,1))`：定义只含一个线性层的模型。
  - 输入形状：`(batch_size, 2)`
  - 输出形状：`(batch_size, 1)`
- 该模型自动管理权重 `w` 和偏置 `b`，可通过 `net.parameters()` 访问。

In [85]:
from torch import nn
net=nn.Sequential(nn.Linear(2,1))

初始化模型参数

In [86]:
net[0].weight.data.normal_(0,std=0.01) # 均值为0，方差为0.01
net[0].bias.data.fill_(0)   

tensor([0.])

计算均方误差使用MSELoss类，也成为平方范数

In [87]:
loss=nn.MSELoss()

## 创建优化器：SGD

- `torch.optim.SGD`：PyTorch 内置的随机梯度下降优化器。
- `net.parameters()`：获取模型中所有可训练参数（权重和偏置）。
- `lr`：学习率，控制更新步长，这里设为 0.03。
- 作用：创建一个优化器对象，后续通过 `step()` 更新参数，`zero_grad()` 清零梯度。
- 典型用法：
  `l.backward()` 计算梯度后，调用 `trainer.step()` 更新参数，再调用 `trainer.zero_grad()` 清零梯度。

In [88]:
trainer=torch.optim.SGD(net.parameters(),lr=0.03)

开始训练模块
## 训练循环逐步骤

**整体结构**  
- 外层循环：控制训练轮数（epoch）。  
- 内层循环：按批次遍历训练数据，每个批次更新一次参数。

**单个批次训练步骤**  
1. `l = loss(net(x), y)`：前向传播，计算预测值与真实标签的损失。  
2. `trainer.zero_grad()`：清零所有参数的梯度，防止累加。  
3. `l.backward()`：反向传播，自动计算损失关于参数的梯度。  
4. `trainer.step()`：优化器根据梯度和学习率更新参数。

**每个 epoch 结束后**  
- 用全量数据计算损失，评估模型整体表现。  
- 打印格式：`epoch 当前轮数, loss 平均损失`。

**关键点**  
- 梯度必须清零，否则会在多个 batch 间累加导致错误。  
- 反向传播会填充 `.grad`，优化器使用 `.grad` 更新参数。  
- 评估阶段不更新参数，只观察损失变化。

In [89]:
num_epoch=3
for epoch in range(num_epoch):
    for x,y in data_iter:
        l=loss(net(x),y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l=loss(net(features),labels)
    print(f'epoch {epoch+1}, loss {l:f}')

epoch 1, loss 0.000257
epoch 2, loss 0.000098
epoch 3, loss 0.000098
